In [1]:
import pandas as pd
df = pd.read_csv("../data/ai_resume_screening.csv")
df.head()

,years_experience,skills_match_score,education_level,project_count,resume_length,github_activity,shortlisted
0,6,84.7,Bachelors,7,234,158,No
1,3,59.1,Masters,5,502,77,No
2,12,100.0,Masters,12,753,381,Yes
3,14,66.8,High School,8,529,407,Yes
4,10,99.6,Bachelors,10,754,331,Yes


In [2]:
df.shape
#The dataset has 30000 rows (resumes), 7 columns (6 features and 1 label)

(30000, 7)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   years_experience    30000 non-null  int64  
 1   skills_match_score  30000 non-null  float64
 2   education_level     30000 non-null  str    
 3   project_count       30000 non-null  int64  
 4   resume_length       30000 non-null  int64  
 5   github_activity     30000 non-null  int64  
 6   shortlisted         30000 non-null  str    
dtypes: float64(1), int64(4), str(2)
memory usage: 1.6 MB


In [4]:
df.describe()

,years_experience,skills_match_score,project_count,resume_length,github_activity
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000
mean,7.506567,73.682653,10.646267,572.584700,325.260667
std,4.624104,16.765909,4.634047,178.709918,159.951803
min,0.000000,0.500000,0.000000,150.000000,0.000000
25%,3.750000,62.100000,7.000000,441.000000,202.000000
50%,7.000000,74.300000,10.000000,574.000000,321.000000
75%,12.000000,86.500000,14.000000,709.000000,443.000000
max,15.000000,100.000000,25.000000,900.000000,842.000000


In [5]:
df.columns

Index(['years_experience', 'skills_match_score', 'education_level',
       'project_count', 'resume_length', 'github_activity', 'shortlisted'],
      dtype='str')

In [6]:
df["shortlisted"].value_counts()
#As we can see, the data is imbalanced (70% yes, 30% no)

shortlisted
Yes    20966
No      9034
Name: count, dtype: int64

In [7]:
df.isnull().sum()
#No missing values

years_experience      0
skills_match_score    0
education_level       0
project_count         0
resume_length         0
github_activity       0
shortlisted           0
dtype: int64

In [8]:
df.duplicated().sum()
#No duplicated rows

np.int64(0)

In [9]:
X = df.drop("shortlisted", axis=1)
y = df["shortlisted"].map({"Yes": 1, "No": 0})

print("Dataset overview")
print(f"Features shape: {X.shape}")
print("Target distribution:")
print(y.value_counts())

Dataset overview
Features shape: (30000, 6)
Target distribution:
shortlisted
1    20966
0     9034
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Data split")
print(f"Train set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.0f}%)")
print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")

Data split
Train set: 21000 samples (70%)
Validation set: 4500 samples (15%)
Test set: 4500 samples (15%)


In [11]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([0, 1])
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

print("Training distribution")
print(f"Negative class (0): {int(y_train.value_counts()[0])}")
print(f"Positive class (1): {int(y_train.value_counts()[1])}")
print(f"Class weights: {class_weight_dict}")

Training distribution
Negative class (0): 6324
Positive class (1): 14676
Class weights: {np.int64(0): np.float64(1.6603415559772297), np.int64(1): np.float64(0.7154538021259199)}


In [12]:
# Education levels have a natural order, so label encoding is more appropriate
education_order = {"High School": 0, "Bachelors": 1, "Masters": 2, "PhD": 3}

X_train_enc = X_train.copy()
X_val_enc   = X_val.copy()
X_test_enc  = X_test.copy()

X_train_enc["education_level"] = X_train["education_level"].map(education_order)
X_val_enc["education_level"]   = X_val["education_level"].map(education_order)
X_test_enc["education_level"]  = X_test["education_level"].map(education_order)

X_train_enc.head()

,years_experience,skills_match_score,education_level,project_count,resume_length,github_activity
13163,1,48.1,1,4,740,99
29921,11,93.7,3,13,836,428
2369,5,77.3,1,10,592,277
5257,12,96.0,3,19,603,558
3640,1,71.6,2,4,509,111


In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_val_scaled   = scaler.transform(X_val_enc)
X_test_scaled  = scaler.transform(X_test_enc)

## Machine Learning Model Selection

Three supervised ML models are trained and compared for this binary classification task: **Decision Tree**, **Random Forest**, and **Logistic Regression**.


### Model 1: Decision Tree


In [14]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=10,
    min_samples_split=40,
    min_samples_leaf=20,
    class_weight=class_weight_dict,
    random_state=55
)
dt.fit(X_train_enc, y_train)

print("Decision Tree trained successfully")
print(f"Criterion: {dt.criterion}")
print(f"Tree depth: {dt.get_depth()}")
print(f"Number of leaves: {dt.get_n_leaves()}")
print(f"Features used: {dt.n_features_in_}")

Decision Tree trained successfully
Criterion: entropy
Tree depth: 10
Number of leaves: 314
Features used: 6


### Model 2: Random Forest


In [15]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, class_weight=class_weight_dict, random_state=42)
rf.fit(X_train_enc, y_train)

print("Random Forest trained successfully")
print(f"Number of trees: {rf.n_estimators}")
print(f"Criterion: {rf.criterion}")
print(f"Features used: {rf.n_features_in_}")

Random Forest trained successfully
Number of trees: 200
Criterion: gini
Features used: 6


### Model 3: Logistic Regression


In [16]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(solver='lbfgs', max_iter=300, C=1.0, class_weight=class_weight_dict, random_state=42)
lr.fit(X_train_scaled, y_train)

print("Logistic Regression trained successfully")
print(f"Solver: {lr.solver}")
print(f"Max iterations: {lr.max_iter}")
print(f"Iterations used: {lr.n_iter_[0]}")
print(f"Regularization C: {lr.C}")
print(f"Features used: {lr.n_features_in_}")

Logistic Regression trained successfully
Solver: lbfgs
Max iterations: 300
Iterations used: 13
Regularization C: 1.0
Features used: 6


## Selected Models

**Logistic Regression** is selected as the primary classification model because its linear architecture and regularization are a strong fit for a low-dimensional tabular dataset with 6 structured features.

**Random Forest** is additionally selected because its ensemble architecture can capture non-linear interactions and provide feature importance scores for candidate feedback.

**Decision Tree** is kept as a simple benchmark because it is easy to interpret, even though it is usually less stable than the other two models.
